# STEP A - SINGLE INSTANCE DETECTION

## Introduction
The provided code performs object detection using the **SIFT** (Scale-Invariant Feature Transform) algorithm and the **FLANN** (Fast Library for Approximate Nearest Neighbors) based matcher. It aims to identify and locate single instances of query images within a set of training images.

## Code Implementation
In the first part, we import all the important libraries, then we load scene and model images and compute their features SIFT.

In [1]:
import numpy as np
import cv2
from matplotlib import pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
scene_paths = ["e1.png","e2.png","e3.png","e4.png","e5.png"]
model_paths = ["0.jpg","1.jpg","11.jpg","19.jpg","24.jpg","26.jpg","25.jpg"]


def load_images(paths,dir):
  return [cv2.cvtColor(cv2.imread("object_detection_project/"+dir+path),cv2.COLOR_BGR2RGB) for path in paths]
  
# Load scene and model images
train_images = load_images(scene_paths,"scenes/")
query_images = load_images(model_paths,"models/")

# Compute keypoints and descriptors
sift = cv2.SIFT_create()

train_features = {index:sift.detectAndCompute(train_image,None) for index,train_image in enumerate(train_images)}
query_features = {index:sift.detectAndCompute(query_image,None) for index,query_image in enumerate(query_images)}

error: OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'


Now we compute matches between query and train descriptors using the Nearest Neighbour Search provided by FLANN and the ratio of distances by Lowe, tuning the threshold _lowe_.

Then, the `get_roi` function extracts a region of interest from the image based on four corner points and the two vertices (upper left and bottom right ones) used to define the bounding box.

Finally, the `print_output` function is designed to annotate and print details about the detected instances of the query image within the training image. It draws bounding boxes around the detected instances and prints their positions and dimensions.

In [ ]:
def compute_matches(FLANN_INDEX_KDTREE,trees,checks,k,lowe,des_query,des_train):
    
  index_params = dict(algorithm = FLANN_INDEX_KDTREE, trees = trees)
  search_params = dict(checks = checks)
  flann = cv2.FlannBasedMatcher(index_params, search_params)
  matches = flann.knnMatch(des_query,des_train,k=2)
    
  # store all the good matches as per Lowe's ratio test.
  good = []
  for m,n in matches:
      if m.distance < lowe*n.distance:
          good.append(m)
          
  return good

def get_roi(x1,x2,x3,x4,y1,y2,y3,y4,img):
    
  top_left_x = int(max(0,min([x1,x2,x3,x4])))
  top_left_y = int(max(0, min([y1,y2,y3,y4])))
  bot_right_x = int(max([x1,x2,x3,x4]))
  bot_right_y = int(max([y1,y2,y3,y4]))
    
  return (img[top_left_y:bot_right_y, top_left_x:bot_right_x],(top_left_x,top_left_y),(bot_right_x,bot_right_y))

def print_output(query_idx,rectangles,img_train):
    
    print("Product {} - {} Instances Found:".format(query_idx,len(rectangles)))
    
    for index in range(len(rectangles)):
        s,e=rectangles[index]
        width = int(e[0]) - int(s[0])
        height = int(e[1]) - int(s[1])
        x_c,y_c=(int((e[0]+s[0])/2),int((e[1]+s[1])/2))
        cv2.rectangle(img_train, np.int32(s),np.int32(e), (0,255,0), thickness = 5)
        cv2.circle(img_train, (int(x_c),int(y_c)), radius=5, color=(0, 255, 0), thickness=-1)
        print("Instance {} [ position: ({},{}), width: {} px, height: {} px]".format(index,x_c,y_c,width,height))

    return len(rectangles)


The `instance_detection` function matches keypoints between query and train images. It computes the homography matrix using RANSAC, validates the matches based on spatial distribution, then it extracts and verifies the ROI based on color differences.

Finally, it draws rectangles around detected instances and annotates them.

# Notes

Since the boxes in the model images are not exactly the same as the ones in the target images, we approached the implementation of the project dividing the images into grids and checking whether there were enough feature matches per cell to declare that the instance was found.

Following this approach, it matches only the boxes that are perfectly the same. 

For this reason, we added a flag _spatial_dist_ to switch between two situations: either it can follow the previous procedure or detect the boxes even if there are little differences. 


In [ ]:
spatial_dist = 1

def instance_detection(query_idx,img_query, img_train, query_features, train_features, min_match_count = 50, COLOR_DIFF_THRESHOLD = 50):
    rectangles=[]
    good = compute_matches(0,5,50,2,0.55,query_features[1],train_features[1])
    
    # If it's a good match, then proceed with the detection
    if len(good)>min_match_count:
        src_pts = np.float32([query_features[0][m.queryIdx].pt for m in good ]).reshape(-1,1,2)
        dst_pts = np.float32([train_features[0][m.trainIdx].pt for m in good ]).reshape(-1,1,2)
        M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 2)
        if M is None:
            print ("No Homography was found")
            
        else:
            h,w,_ = img_query.shape
            if spatial_dist:
                #Counting and checking whether there are enough feature matches per cell to declare that the instance was found 
                grid_size = 3  # grid dimension (4x4) 
                grid_counts = np.zeros((grid_size, grid_size), dtype=int)
            
                for pt in src_pts:
                    x, y = pt[0]
                    grid_x = int(x // (w / grid_size))
                    grid_y = int(y // (h / grid_size))
                    grid_counts[grid_y, grid_x] += 1
                
                min_matches_per_cell = 3 #minimum number of matches
                valid_cells = np.sum(grid_counts >= min_matches_per_cell)
            
                if valid_cells >= 0.75 * (grid_size * grid_size): #checks if at least 75% of the cells have a minimum number of matches
                # Generate and plot the rectangle into the target image
                    pts = np.float32([ [0,0],[0,h-1],[w-1,h-1],[w-1,0] ]).reshape(-1,1,2)
                    dst = cv2.perspectiveTransform(pts,M)
                    roi,start,end= get_roi(dst[0][0][0],dst[1][0][0],dst[2][0][0],dst[3][0][0],dst[0][0][1],dst[1][0][1],dst[2][0][1],dst[3][0][1],img_train)
                    # If we can manage to obtain a rectangle, then we proceed
                    if len(roi):
                      # check if the euclidean distance between the colors (mean) of the two rectangle is lower than the chosen threshold
                      color_diff = np.linalg.norm(roi.mean(axis = 0).mean(axis = 0) - img_query.mean(axis = 0).mean(axis = 0))
                      if color_diff <= COLOR_DIFF_THRESHOLD:
                        rectangles.append((start,end))
            else:
                 pts = np.float32([ [0,0],[0,h-1],[w-1,h-1],[w-1,0] ]).reshape(-1,1,2)
                 dst = cv2.perspectiveTransform(pts,M)
                 roi,start,end= get_roi(dst[0][0][0],dst[1][0][0],dst[2][0][0],dst[3][0][0],dst[0][0][1],dst[1][0][1],dst[2][0][1],dst[3][0][1],img_train)
                 # If we can manage to obtain a rectangle, then we proceed
                 if len(roi):
                     # check if the euclidean distance between the colors (mean) of the two rectangle is lower than the chosen threshold
                     color_diff = np.linalg.norm(roi.mean(axis = 0).mean(axis = 0) - img_query.mean(axis = 0).mean(axis = 0))
                     if color_diff <= COLOR_DIFF_THRESHOLD:
                         rectangles.append((start,end))
            
    if len(rectangles):
      print_output(query_idx,rectangles,img_train)

    return len(rectangles)

# Iterating among all scenes and looking for the query object.
for index_train  in range(len(train_images)):
    
  for index_query in range(len(query_images)):
    instance_detection(index_query,query_images[index_query], train_images[index_train], query_features[index_query], train_features[index_train])

#Visualization
  plt.imshow(train_images[index_train])
  plt.show()
